In [ ]:
!pip install pandas openpyxl groq

In [ ]:
# Décommentez si nécessaire
# !pip install pandas numpy scikit-learn matplotlib seaborn -q

import sqlite3
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)


In [ ]:
SQL_PATH = 'universite_dw.sql'

with open(SQL_PATH, encoding='utf-8') as f:
    sql = f.read()

# MySQL échappe les quotes avec un backslash (\'), SQLite attend un doublement ('')
sql = sql.replace("\\'", "''")

creates = re.findall(r'CREATE TABLE `.*?;\n', sql, re.S)
creates = [re.sub(r'\)\s*ENGINE=\w+[^;]*;', ');', c) for c in creates]
inserts = re.findall(r'INSERT INTO `.*?;\n', sql, re.S)

conn = sqlite3.connect(':memory:')
cur = conn.cursor()
for stmt in creates:
    cur.execute(stmt)
for stmt in inserts:
    cur.execute(stmt)
conn.commit()

print(f"{len(creates)} tables créées, {len(inserts)} lots de données insérés.")
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
print([r[0] for r in cur.fetchall()])


16 tables créées, 120 lots de données insérés.
['dim_category', 'dim_classement_arwu_shanghai', 'dim_classement_qs_monde', 'dim_classement_the_impact', 'dim_classement_the_monde', 'dim_classement_universite', 'dim_country', 'dim_date', 'dim_durabilite', 'dim_rapports_annuels', 'dim_rapports_details', 'dim_stars_credits', 'dim_stars_scores', 'dim_universite', 'dim_universite_details', 'fact_universite']


In [ ]:
# ============================================================
# 🤖 CHATBOT BI — DATABASE SQLITE + GROQ
# Google Colab
# ============================================================

# ─── Installation ────────────────────────────────────────────
# !pip install -q pandas groq openpyxl


# ─── Imports ─────────────────────────────────────────────────

import re
import sqlite3
import pandas as pd
from getpass import getpass
from groq import Groq


# ============================================================
# 1. CONFIGURATION GROQ
# ============================================================

print("=" * 60)
print("🔑 Configuration Groq")
print("=" * 60)

GROQ_API_KEY = getpass("Entrez votre Groq API Key : ")

MODEL = "openai/gpt-oss-120b"

client = Groq(
    api_key=GROQ_API_KEY
)


# ============================================================
# 2. CHARGEMENT DE LA DATABASE SQLITE
# ============================================================

# IMPORTANT :
# Cette partie suppose que tu as déjà exécuté ton code
# qui transforme universite_dw.sql en database SQLite.
#
# Tu dois déjà avoir :
#
# conn = sqlite3.connect(':memory:')
#
# et les 16 tables chargées dedans.


# Vérifier que conn existe
try:
    conn
except NameError:
    raise Exception(
        "❌ La connexion SQLite 'conn' n'existe pas. "
        "Exécute d'abord le code de chargement de universite_dw.sql."
    )


# ============================================================
# 3. RÉCUPÉRER LES TABLES
# ============================================================

cur = conn.cursor()

cur.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""")

tables = [
    row[0]
    for row in cur.fetchall()
]


print("\n" + "=" * 60)
print("🗄️ DATABASE")
print("=" * 60)

print(f"\n✅ Nombre de tables : {len(tables)}")

for table in tables:
    print("   -", table)


# ============================================================
# 4. CHARGER TOUTES LES TABLES DANS UN DICTIONNAIRE
# ============================================================

database = {}

print("\n" + "=" * 60)
print("📊 CONTENU DES TABLES")
print("=" * 60)

for table in tables:

    try:

        df_table = pd.read_sql_query(
            f'SELECT * FROM "{table}"',
            conn
        )

        database[table] = df_table

        print(
            f"\n📋 {table}"
            f"\n   → Lignes    : {df_table.shape[0]:,}"
            f"\n   → Colonnes  : {df_table.shape[1]}"
            f"\n   → Colonnes  : {list(df_table.columns)}"
        )

    except Exception as e:

        print(
            f"\n❌ Erreur avec la table {table} : {e}"
        )


# ============================================================
# 5. CONSTRUIRE LE SCHÉMA DE LA DATABASE
# ============================================================

def construire_schema(conn):

    cur = conn.cursor()

    cur.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
    """)

    tables = [
        row[0]
        for row in cur.fetchall()
    ]

    schema = []

    for table in tables:

        schema.append(
            f"\nTABLE: {table}"
        )

        # ----------------------------------------------------
        # Colonnes
        # ----------------------------------------------------

        cur.execute(
            f'PRAGMA table_info("{table}")'
        )

        columns = cur.fetchall()

        for col in columns:

            # PRAGMA table_info:
            # cid, name, type, notnull, default, pk

            column_name = col[1]
            column_type = col[2]
            primary_key = col[5]

            line = (
                f"- {column_name} "
                f"({column_type})"
            )

            if primary_key == 1:
                line += " [PRIMARY KEY]"

            schema.append(line)

        # ----------------------------------------------------
        # Foreign Keys
        # ----------------------------------------------------

        cur.execute(
            f'PRAGMA foreign_key_list("{table}")'
        )

        foreign_keys = cur.fetchall()

        for fk in foreign_keys:

            # SQLite:
            # id, seq, table, from, to,
            # on_update, on_delete, match

            referenced_table = fk[2]
            from_column = fk[3]
            to_column = fk[4]

            schema.append(
                f"- FK: {table}.{from_column} "
                f"-> {referenced_table}.{to_column}"
            )

    return "\n".join(schema)


SCHEMA = construire_schema(conn)


# ============================================================
# 6. RELATIONS IMPORTANTES DU DATA WAREHOUSE
# ============================================================

RELATIONS = """

RELATIONS IMPORTANTES DU DATA WAREHOUSE :

fact_universite.classement_arwu_shanghai_fk
    -> dim_classement_arwu_shanghai.id

fact_universite.classement_qs_monde_fk
    -> dim_classement_qs_monde.id

fact_universite.classement_the_impact_fk
    -> dim_classement_the_impact.id

fact_universite.classement_the_monde_fk
    -> dim_classement_the_monde.id

fact_universite.country_fk
    -> dim_country.id

fact_universite.date_fk
    -> dim_date.id

fact_universite.university_fk
    -> dim_universite.university_id

dim_universite.universite_informations_fk
    -> dim_universite_details.id

dim_stars_credits.category_fk
    -> dim_category.id

dim_stars_scores.category_fk
    -> dim_category.id

dim_rapports_annuels.details_rapports_fk
    -> dim_rapports_details.id
"""


DATABASE_CONTEXT = SCHEMA + RELATIONS


# ============================================================
# 7. AFFICHER LE SCHÉMA
# ============================================================

print("\n" + "=" * 60)
print("🧩 SCHÉMA DE LA DATABASE")
print("=" * 60)

print(DATABASE_CONTEXT)


# ============================================================
# 8. FONCTION DE NETTOYAGE DU SQL
# ============================================================

def nettoyer_sql(sql):

    if sql is None:
        raise ValueError(
            "Le modèle n'a retourné aucune requête SQL."
        )

    sql = sql.strip()

    # Supprimer les blocs Markdown
    sql = re.sub(
        r"```sql",
        "",
        sql,
        flags=re.IGNORECASE
    )

    sql = re.sub(
        r"```",
        "",
        sql
    )

    sql = sql.strip()

    # Chercher SELECT ou WITH
    match = re.search(
        r"\b(SELECT|WITH)\b",
        sql,
        flags=re.IGNORECASE
    )

    if not match:

        raise ValueError(
            "Aucune requête SELECT/WITH trouvée dans la réponse du modèle.\n\n"
            f"Réponse reçue :\n{sql}"
        )

    # Commencer à partir de SELECT/WITH
    sql = sql[
        match.start():
    ].strip()

    # Supprimer ; final
    sql = sql.rstrip(";").strip()

    # Vérifier que SELECT ou WITH est bien au début
    if not re.match(
        r"^(SELECT|WITH)\b",
        sql,
        flags=re.IGNORECASE
    ):

        raise ValueError(
            "La requête générée n'est pas une requête SELECT/WITH."
        )

    # --------------------------------------------------------
    # Sécurité SQL
    # --------------------------------------------------------

    forbidden = re.compile(
        r"\b("
        r"INSERT|"
        r"UPDATE|"
        r"DELETE|"
        r"DROP|"
        r"ALTER|"
        r"CREATE|"
        r"REPLACE|"
        r"TRUNCATE|"
        r"ATTACH|"
        r"DETACH|"
        r"PRAGMA|"
        r"VACUUM"
        r")\b",
        flags=re.IGNORECASE
    )

    if forbidden.search(sql):

        raise ValueError(
            "❌ Requête SQL interdite détectée."
        )

    # Une seule requête
    if ";" in sql:

        raise ValueError(
            "❌ Plusieurs requêtes SQL ne sont pas autorisées."
        )

    return sql


# ============================================================
# 9. GÉNÉRER UNE REQUÊTE SQL AVEC GROQ
# ============================================================

def generer_sql(question, historique):

    # Garder seulement quelques messages
    historique_recent = historique[-4:]

    conversation = ""

    if historique_recent:

        conversation = "\n\nCONVERSATION RÉCENTE :\n"

        for message in historique_recent:

            role = message["role"]
            content = message["content"]

            conversation += (
                f"\n{role.upper()} : {content}"
            )


    prompt = f"""
Tu es un expert en SQL, Business Intelligence
et analyse de données universitaires.

Tu travailles avec une base de données SQLITE.

============================================================
STRUCTURE DE LA DATABASE
============================================================

{DATABASE_CONTEXT}

{conversation}

============================================================
QUESTION DE L'UTILISATEUR
============================================================

{question}

============================================================
RÈGLES
============================================================

1. Génère UNE SEULE requête SQL.

2. La requête doit être compatible avec SQLITE.

3. Utilise uniquement les tables et colonnes
   présentes dans le schéma.

4. Tu peux utiliser plusieurs tables avec JOIN.

5. Utilise les relations du Data Warehouse fournies
   ci-dessus.

6. Ne crée jamais une colonne ou une table inexistante.

7. Utilise uniquement SELECT ou WITH.

8. INTERDIT :
   INSERT
   UPDATE
   DELETE
   DROP
   ALTER
   CREATE
   REPLACE
   TRUNCATE
   ATTACH
   DETACH
   PRAGMA
   VACUUM

9. Pour une question demandant une liste,
   utilise LIMIT 100.

10. Pour COUNT, AVG, SUM, MIN, MAX, etc.,
    effectue directement le calcul dans SQL.

11. Si l'utilisateur demande une comparaison
    entre plusieurs années, utilise la colonne year
    appropriée.

12. Si l'utilisateur demande un classement,
    utilise la colonne de classement appropriée.

13. Les classements disponibles sont :
    - ARWU → dim_classement_arwu_shanghai
    - QS → dim_classement_qs_monde
    - THE → dim_classement_the_monde
    - THE Impact → dim_classement_the_impact

14. Si la question est ambiguë, utilise les données
    disponibles et fais une réponse basée uniquement
    sur les informations présentes.

15. Retourne UNIQUEMENT la requête SQL.

16. Ne mets pas de ```sql.

17. Ne donne aucune explication.

============================================================
SQL :
============================================================
"""


    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": (
                    "Tu es un expert SQL SQLite. "
                    "Tu génères uniquement des requêtes "
                    "SQL analytiques en lecture seule."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        max_tokens=700,

        temperature=0
    )


    sql_brut = response.choices[0].message.content

    return nettoyer_sql(sql_brut)


# ============================================================
# 10. EXÉCUTER LA REQUÊTE SQL
# ============================================================

def executer_sql(sql):

    try:

        df_resultat = pd.read_sql_query(
            sql,
            conn
        )

        return df_resultat

    except Exception as e:

        print("\n❌ Erreur SQL :")
        print(e)

        print("\n🔎 SQL problématique :")
        print(sql)

        return None


# ============================================================
# 11. GÉNÉRER LA RÉPONSE FINALE
# ============================================================

def generer_reponse(
    question,
    sql,
    df_resultat
):

    if df_resultat is None:

        return (
            "Une erreur est survenue "
            "lors de l'exécution de la requête."
        )


    if df_resultat.empty:

        return (
            "Aucune donnée ne correspond "
            "à votre question."
        )


    # Maximum 100 lignes envoyées à Groq
    resultat = df_resultat.head(100)


    resultat_texte = resultat.to_string(
        index=False
    )


    prompt = f"""
Tu es un assistant BI spécialisé
dans l'analyse de données universitaires.

Réponds à la question de l'utilisateur
en utilisant UNIQUEMENT le résultat SQL fourni.

============================================================
QUESTION
============================================================

{question}

============================================================
REQUÊTE SQL
============================================================

{sql}

============================================================
RÉSULTAT
============================================================

{resultat_texte}

============================================================
RÈGLES
============================================================

- Réponds en français.
- Réponds clairement et naturellement.
- Utilise les chiffres présents dans le résultat.
- Ne fabrique aucune information.
- Ne fais pas d'affirmation qui n'est pas supportée
  par le résultat.
- Si plusieurs lignes existent, présente-les clairement.
- Tu peux utiliser un tableau Markdown.
- Si le résultat est vide, indique-le.
- Ne parle pas de Groq.
- Ne parle pas de SQLite.
- Ne parle pas de la génération de la requête SQL.
"""


    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": (
                    "Tu es un assistant BI. "
                    "Réponds uniquement à partir "
                    "des résultats fournis."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        max_tokens=700,

        temperature=0.2
    )


    return response.choices[0].message.content


# ============================================================
# 12. CHATBOT
# ============================================================

def lancer_chatbot(conn):

    print("\n" + "=" * 60)
    print("  🤖 Chatbot BI — Groq + GPT-OSS 120B + SQLite")
    print("=" * 60)

    print(
        f"\n🗄️ {len(tables)} tables disponibles."
    )

    print(
        "\n💬 Pose n'importe quelle question "
        "sur ta base de données !"
    )

    print(
        "   Écris 'quitter' pour arrêter."
    )

    print("\n" + "-" * 60)


    historique = []


    while True:

        question = input(
            "\nVous : "
        ).strip()


        if not question:
            continue


        if question.lower() in [
            "quitter",
            "exit",
            "quit",
            "q"
        ]:

            print(
                "\nAu revoir ! 👋"
            )

            break


        try:

            # ==================================================
            # ÉTAPE 1
            # Question → SQL
            # ==================================================

            print(
                "\n⏳ Analyse de la question..."
            )


            sql = generer_sql(
                question,
                historique
            )


            # ==================================================
            # ÉTAPE 2
            # Afficher SQL
            # ==================================================

            print(
                "\n🔎 SQL généré :"
            )

            print("-" * 60)

            print(sql)


            # ==================================================
            # ÉTAPE 3
            # SQL → Database
            # ==================================================

            df_resultat = executer_sql(
                sql
            )


            if df_resultat is None:

                continue


            # ==================================================
            # ÉTAPE 4
            # Afficher résultat
            # ==================================================

            print(
                "\n📊 Résultat :"
            )

            print("-" * 60)

            display(
                df_resultat.head(20)
            )


            # ==================================================
            # ÉTAPE 5
            # Résultat → réponse naturelle
            # ==================================================

            print(
                "\n⏳ Génération de la réponse..."
            )


            reponse = generer_reponse(
                question,
                sql,
                df_resultat
            )


            print(
                "\n🤖 Bot :"
            )

            print(reponse)


            # ==================================================
            # HISTORIQUE
            # ==================================================

            historique.append({
                "role": "user",
                "content": question
            })


            historique.append({
                "role": "assistant",
                "content": reponse
            })


            # Garder seulement les 6 derniers messages
            if len(historique) > 6:

                historique = historique[-6:]


        except Exception as e:

            print(
                f"\n❌ Erreur : {e}"
            )


# ============================================================
# 13. LANCER LE CHATBOT
# ============================================================

lancer_chatbot(conn)

🔑 Configuration Groq
Entrez votre Groq API Key : ··········

🗄️ DATABASE

✅ Nombre de tables : 16
   - dim_category
   - dim_classement_arwu_shanghai
   - dim_classement_qs_monde
   - dim_classement_the_impact
   - dim_classement_the_monde
   - dim_classement_universite
   - dim_country
   - dim_date
   - dim_durabilite
   - dim_rapports_annuels
   - dim_rapports_details
   - dim_stars_credits
   - dim_stars_scores
   - dim_universite
   - dim_universite_details
   - fact_universite

📊 CONTENU DES TABLES

📋 dim_category
   → Lignes    : 5
   → Colonnes  : 3
   → Colonnes  : ['id', 'categorie', 'categorie_encoded']

📋 dim_classement_arwu_shanghai
   → Lignes    : 210
   → Colonnes  : 7
   → Colonnes  : ['id', 'name', 'country', 'year', 'arwu_rank', 'arwu_score', 'lien']

📋 dim_classement_qs_monde
   → Lignes    : 1,503
   → Colonnes  : 9
   → Colonnes  : ['id', 'name', 'country', 'year', 'qs_rank', 'qs_overall', 'qs_employer_reputation', 'qs_employment_outcomes', 'lien']

📋 dim_classeme

KeyboardInterrupt: Interrupted by user